# LOCOS Demo

This notebook walks through the standalone LOCOS workflow with CPU-safe artifact inspection first and GPU-required detection/ablation commands clearly marked.

In [ ]:
from pathlib import Path

repo = Path.cwd()
print(repo)

Download NoLiMa probing data from the command line:

```bash
make data DATASET=nolima
```

In [ ]:
sample_scores = {
    "0-0": [0.02, 0.04, 0.03],
    "0-1": [0.24, 0.21, 0.27],
    "1-0": [-0.03, -0.01, 0.00],
    "1-1": [0.18, 0.20, 0.17],
}
ranked = sorted(
    ((head, sum(vals) / len(vals)) for head, vals in sample_scores.items()),
    key=lambda item: item[1],
    reverse=True,
)
ranked[:3]

In [ ]:
import matplotlib.pyplot as plt

heads, scores = zip(*ranked)
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.bar(heads, scores, color=["#d84a2b", "#2672b9", "#f2c84b", "#222222"])
ax.set_ylabel("Mean score")
ax.set_title("Example LOCOS head scores")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

GPU-required detection:

```bash
make detect MODEL=Qwen/Qwen3-8B DATASET=nolima ARGS="--num-examples 8"
```

GPU-required mean-ablation eval:

```bash
make ablate MODEL=Qwen/Qwen3-8B HEADS=retrieval_heads/Qwen3-8B_logit_contrib_nolima.json ARGS="--num-examples 20"
```

In [ ]:
from locos_eval import load_retrieval_heads

heads_path = repo / "retrieval_heads" / "Qwen3-8B_logit_contrib_nolima.json"
if heads_path.exists():
    print(load_retrieval_heads(heads_path, num_heads=5))
else:
    print(f"No local head file yet: {heads_path}")